# Building native graphs

A graph specifies components, their ports, and the wires between them. An instant wire carries a value within the current step. A recurrent wire carries the previous step's value and declares where its initial value comes from.

In [ ]:
import jax.numpy as jnp
import jax.random as jr

from feedbax.compiler import GraphDocument, compile_graph
from feedbax.compiler.templates import simple_feedback_template_graph
from feedbax.component_registry import ComponentRegistry
from feedbax.contracts.graph import GraphSpec

spec = simple_feedback_template_graph({
    "network": {"input_size": 8, "hidden_size": 12, "out_size": 2},
    "feedback_delay": 2,
})
[(wire.source_node, wire.target_node, wire.temporality) for wire in spec.wires]

## Compile and run

GraphSpec is portable JSON. Compiling it creates the same kind of graph used by the Python model constructors and Studio. The graph below takes two external input features and six features selected from the point mass state.

In [ ]:
saved = spec.model_dump_json()
restored = GraphSpec.model_validate_json(saved)
registry = ComponentRegistry(load_user_components=False)
model = compile_graph(GraphDocument(graph=restored), registry).graph

key = jr.key(0)
outputs, _, history = model(
    {"input": jnp.ones((4, 2))},
    model.init_state(key=key),
    n_steps=4,
    key=key,
    return_state_history=True,
)
outputs["effector"].pos.shape, history.net.hidden.shape

## Observe a port

Retain the controller's output directly to observe its command before the motor channel adds noise or delay. The port trace contains one observation per transition. State history also includes the initial sample.

In [ ]:
from feedbax.runtime.graph import GraphTraceRequest

_, _, trace = model.rollout_with_trace(
    {"input": jnp.ones((4, 2))},
    model.init_state(key=key),
    key=key,
    n_steps=4,
    trace=(GraphTraceRequest(kind="port", selector="command", node="net", port="output"),),
)
trace["command"].shape